In [1]:
import torch
import numpy as np

# ============================================
# Initialize PyTorch with MPS (Apple Silicon GPU)
# ============================================

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✓ MPS (Apple Silicon GPU) is available and will be used!")
else:
    device = torch.device("cpu")
    print("⚠ MPS not available, falling back to CPU")

print(f"Device: {device}")
print(f"PyTorch Version: {torch.__version__}\n")

✓ MPS (Apple Silicon GPU) ist verfügbar und wird verwendet!
Device: mps
PyTorch Version: 2.9.0



In [2]:
try:
    from datasets import load_dataset
    
    # Option 1: German Wikipedia (good quality, medium-sized)
    dataset_hf = load_dataset("wikimedia/wikipedia", "20231101.de", split="train", streaming=False)
    
    # Take only the first N articles for faster training
    NUM_ARTICLES = 1000  # Feel free to increase this!
    dataset_hf = dataset_hf.select(range(min(NUM_ARTICLES, len(dataset_hf))))
    
    # Combine all texts
    print(f"Processing {len(dataset_hf)} Wikipedia articles...")
    texts = [article['text'] for article in dataset_hf]
    text = ' '.join(texts)
    
    print(f"✓ Dataset loaded!")
    print(f"  Articles: {NUM_ARTICLES}")
    print(f"  Characters: {len(text):,}")
    
except ImportError:
    print("⚠ 'datasets' not installed. Install it with: pip install datasets")
    print("Using fallback text...\n")
    # German fallback corpus - the model is trained on German text
    text = "Transformer sind Deep-Learning-Modelle. " * 1000
except Exception as e:
    print(f"⚠ Error while loading: {e}")
    print("Using fallback text...\n")
    # German fallback corpus - the model is trained on German text
    text = "Transformer sind Deep-Learning-Modelle. " * 1000

/Users/bertram/Documents/transfomer-test/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Verarbeite 1000 Wikipedia-Artikel...
✓ Dataset geladen!
  Artikel: 1000
  Zeichen: 23,541,885


In [3]:
import tiktoken

encoding = tiktoken.get_encoding("gpt2")

# Tokenize the text
tokens = encoding.encode(text)

print(f"✓ Text tokenized!")
print(f"  Number of tokens: {len(tokens)}")
print(f"  Vocab Size: {encoding.n_vocab}")
print(f"\nFirst 20 token IDs: {tokens[:20]}")

# Back to text (just for testing)
decoded_sample = encoding.decode(tokens[:20])
print(f"\nFirst 20 tokens decoded:\n'{decoded_sample}'")

# Tokens as a PyTorch tensor
token_tensor = torch.tensor(tokens, device=device)
print(f"\n✓ Tokens turned into a tensor!")
print(f"  Shape: {token_tensor.shape}")
print(f"  Device: {token_tensor.device}")

✓ Text tokenisiert!
  Anzahl Tokens: 8843709
  Vocab Size: 50257

Erste 20 Token IDs: [36235, 2439, 270, 21067, 2876, 4352, 435, 82, 49693, 463, 5177, 277, 25151, 304, 42326, 277, 1134, 83, 1469, 3310]

Dekodierte erste 20 Tokens:
'Alan Smithee steht als Pseudonym für einen fiktiven Reg'

✓ Tokens als Tensor erstellt!
  Shape: torch.Size([8843709])
  Device: mps:0


In [4]:
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    """
    Dataset for language modeling:
    Input: Tokens [0, 1, 2, 3, 4]
    Target: Tokens [1, 2, 3, 4, 5]
    """
    def __init__(self, tokens, seq_length=128):
        self.tokens = tokens
        self.seq_length = seq_length
        self.num_sequences = (len(tokens) - 1) // seq_length
        
    def __len__(self):
        return self.num_sequences
    
    def __getitem__(self, idx):
        start_idx = idx * self.seq_length
        end_idx = start_idx + self.seq_length + 1
        
        sequence = self.tokens[start_idx:end_idx]
        
        # Input: all tokens except the last one
        input_ids = torch.tensor(sequence[:-1], dtype=torch.long)
        # Target: all tokens except the first one
        target_ids = torch.tensor(sequence[1:], dtype=torch.long)
        
        return input_ids, target_ids

In [5]:

SEQ_LENGTH = 128  # Length of each sequence
BATCH_SIZE = 8  # Number of sequences per batch

dataset = TextDataset(tokens, seq_length=SEQ_LENGTH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print(f"✓ DataLoader recreated: {len(dataloader)} batches")

✓ DataLoader neu erstellt: 8636 Batches


In [6]:
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    """
    Adds position information to the embeddings.
    Without this, transformers have no notion of word order!
    """
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        
        # Build the positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        
        # Compute the sinusoids
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                            -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)  # Even indices
        pe[:, 1::2] = torch.cos(position * div_term)  # Odd indices
        
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """
        x: [batch_size, seq_len, d_model]
        """
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]

In [7]:
class TokenEmbedding(nn.Module):
    """
    Combines token embeddings with positional encoding
    """
    def __init__(self, vocab_size, d_model, max_len=5000):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.d_model = d_model
        
    def forward(self, x):
        """
        x: [batch_size, seq_len] - Token IDs
        Returns: [batch_size, seq_len, d_model] - Embeddings
        """
        # Token IDs -> vectors
        x = self.embedding(x) * math.sqrt(self.d_model)
        
        # Add positional encoding
        x = self.pos_encoding(x)
        
        return x

In [8]:
VOCAB_SIZE = encoding.n_vocab  # ~50257 for GPT-2
D_MODEL = 512

# Create the embedding layer
embedding_layer = TokenEmbedding(VOCAB_SIZE, D_MODEL).to(device)

print(f"✓ Embedding layer created!")
print(f"  Vocab Size: {VOCAB_SIZE}")
print(f"  Embedding Dimension: {D_MODEL}")
print(f"  Parameters: {sum(p.numel() for p in embedding_layer.parameters()):,}")



✓ Embedding Layer erstellt!
  Vocab Size: 50257
  Embedding Dimension: 512
  Parameter: 25,731,584


In [9]:
import torch.nn.functional as F

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    The core formula: Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) * V
    
    Args:
        Q: Query [batch, seq_len, d_k]
        K: Key [batch, seq_len, d_k]
        V: Value [batch, seq_len, d_k]
        mask: Optional [batch, seq_len, seq_len]
    
    Returns:
        output: [batch, seq_len, d_k]
        attention_weights: [batch, seq_len, seq_len]
    """
    d_k = Q.size(-1)
    
    # Step 1: QK^T - "similarity" between query and key
    scores = torch.matmul(Q, K.transpose(-2, -1))  # [batch, seq_len, seq_len]
    
    # Step 2: Scale by sqrt(d_k)
    scores = scores / math.sqrt(d_k)
    
    # Step 3: Optional - mask (e.g. for causal attention)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    # Step 4: Softmax - turn scores into probabilities
    attention_weights = F.softmax(scores, dim=-1)
    
    # Step 5: Weighted sum of the values
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

In [10]:
class MultiHeadAttention(nn.Module):
    """
    Multi-head attention: several attention heads working in parallel
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads!"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Dimension per head
        
        # Linear projections for Q, K, V (for ALL heads at once)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        # Output projection (after concatenating the heads)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        
    def split_heads(self, x):
        """
        Splits d_model into num_heads and d_k
        
        Input:  [batch, seq_len, d_model]
        Output: [batch, num_heads, seq_len, d_k]
        """
        batch_size, seq_len, d_model = x.size()
        
        # Reshape: [batch, seq_len, num_heads, d_k]
        x = x.view(batch_size, seq_len, self.num_heads, self.d_k)
        
        # Transpose: [batch, num_heads, seq_len, d_k]
        return x.transpose(1, 2)
    
    def combine_heads(self, x):
        """
        Concatenates all heads back together
        
        Input:  [batch, num_heads, seq_len, d_k]
        Output: [batch, seq_len, d_model]
        """
        batch_size, num_heads, seq_len, d_k = x.size()
        
        # Transpose: [batch, seq_len, num_heads, d_k]
        x = x.transpose(1, 2)
        
        # Reshape: [batch, seq_len, d_model]
        return x.contiguous().view(batch_size, seq_len, self.d_model)
    
    def forward(self, x, mask=None):
        """
        x: [batch, seq_len, d_model]
        Returns: [batch, seq_len, d_model], attention_weights
        """
        batch_size = x.size(0)
        
        # 1. Linear projections for Q, K, V
        Q = self.W_q(x)  # [batch, seq_len, d_model]
        K = self.W_k(x)  # [batch, seq_len, d_model]
        V = self.W_v(x)  # [batch, seq_len, d_model]
        
        # 2. Split into multiple heads
        Q = self.split_heads(Q)  # [batch, num_heads, seq_len, d_k]
        K = self.split_heads(K)  # [batch, num_heads, seq_len, d_k]
        V = self.split_heads(V)  # [batch, num_heads, seq_len, d_k]
        
        # 3. Scaled dot-product attention for all heads in parallel
        attention_output, attention_weights = scaled_dot_product_attention(Q, K, V, mask)
        # attention_output: [batch, num_heads, seq_len, d_k]
        
        # 4. Concatenate all heads
        attention_output = self.combine_heads(attention_output)
        # attention_output: [batch, seq_len, d_model]
        
        # 5. Final linear projection
        output = self.W_o(attention_output)
        
        # 6. Dropout
        output = self.dropout(output)
        
        return output, attention_weights


In [11]:
class FeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network:
    FFN(x) = max(0, xW1 + b1)W2 + b2
    
    d_model -> d_ff -> d_model
    (512 -> 2048 -> 512)
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        x = self.linear1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        x = self.dropout(x)
        return x

In [12]:
class DecoderBlock(nn.Module):
    """
    Transformer decoder block - with MASKED attention!
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        
        # Masked Multi-Head Attention
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        # Masked Attention + Residual + Norm
        attention_output, _ = self.attention(x, mask)
        x = self.norm1(x + self.dropout(attention_output))
        
        # Feed-Forward + Residual + Norm
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        
        return x

In [13]:
class GPTDecoder(nn.Module):
    """
    GPT-style transformer decoder (decoder-only architecture)
    """
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        
        self.embedding = TokenEmbedding(vocab_size, d_model)
        
        self.decoder_blocks = nn.ModuleList([
            DecoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        # Language Model Head: d_model -> vocab_size
        self.lm_head = nn.Linear(d_model, vocab_size)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        """
        x: [batch, seq_len] - Token IDs
        Returns: [batch, seq_len, vocab_size] - logits for every token
        """
        x = self.embedding(x)
        x = self.dropout(x)
        
        # Pass through all decoder blocks
        for decoder_block in self.decoder_blocks:
            x = decoder_block(x, mask)
        
        # Language Model Head
        logits = self.lm_head(x)
        
        return logits
    
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0):
        """
        Generate text (autoregressively)
        """
        self.eval()
        
        for _ in range(max_new_tokens):
            # Create the causal mask
            seq_len = input_ids.size(1)
            mask = create_causal_mask(seq_len, input_ids.device)
            
            # Forward pass
            with torch.no_grad():
                logits = self.forward(input_ids, mask)
            
            # Take the last token
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            
            # Sample the next token
            next_token = torch.multinomial(probs, num_samples=1)
            
            # Append it to the sequence
            input_ids = torch.cat([input_ids, next_token], dim=1)
        
        return input_ids

In [14]:
NUM_HEADS = 8
D_FF = 2048
NUM_LAYERS = 6

model = GPTDecoder(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    d_ff=D_FF,
    num_layers=NUM_LAYERS,
    dropout=0.1
).to(device)

print(f"✓ GPT decoder created!")
print(f"  d_model: {D_MODEL}")
print(f"  num_heads: {NUM_HEADS}")
print(f"  d_ff: {D_FF}")
print(f"  num_layers: {NUM_LAYERS}")
print(f"  vocab_size: {VOCAB_SIZE}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

✓ GPT Decoder erstellt!
  d_model: 512
  num_heads: 8
  d_ff: 2048
  num_layers: 6
  vocab_size: 50257
  Parameter: 70,427,729


In [15]:
def create_causal_mask(seq_len, device):
    """
    Creates a mask so that position i can only attend to positions <= i.
    Prevents the decoder from looking into the future!
    
    Returns: [seq_len, seq_len] with 1 for allowed, 0 for forbidden
    """
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask  # Lower triangular matrix


In [16]:
def save_checkpoint(model, optimizer, epoch, loss, filepath):
    """
    Saves model + optimizer + training info
    """
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
        'vocab_size': VOCAB_SIZE,
        'd_model': D_MODEL,
        'num_heads': NUM_HEADS,
        'd_ff': D_FF,
        'num_layers': NUM_LAYERS,
    }
    torch.save(checkpoint, filepath)
    print(f"  💾 Checkpoint saved: {filepath}")


def load_checkpoint(filepath, model, optimizer=None):
    """
    Loads the model + optionally the optimizer
    """
    checkpoint = torch.load(filepath, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']
    
    print(f"  📂 Checkpoint loaded: epoch {epoch}, loss {loss:.4f}")
    return epoch, loss

In [17]:
criterion = nn.CrossEntropyLoss()

# Optimizer: Adam
learning_rate = 3e-4
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

epoch, loss = load_checkpoint('checkpoint_epoch_2.pt', model, optimizer)

print(f"✓ Loss: CrossEntropyLoss")
print(f"✓ Optimizer: Adam (lr={learning_rate})")

# ============================================
# 13. TRAINING LOOP
# ============================================

def train_epoch(model, dataloader, criterion, optimizer, device):
    """
    Trains for one epoch
    """
    model.train()
    total_loss = 0
    
    for batch_idx, (input_ids, target_ids) in enumerate(dataloader):
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)
        
        # Causal mask
        seq_len = input_ids.size(1)
        mask = create_causal_mask(seq_len, device)
        
        # Forward pass
        logits = model(input_ids, mask)
        
        # Compute the loss
        # logits: [batch, seq_len, vocab_size]
        # target: [batch, seq_len]
        loss = criterion(logits.view(-1, VOCAB_SIZE), target_ids.view(-1))
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Progress
        if (batch_idx + 1) % 10 == 0:
            print(f"  Batch {batch_idx + 1}/{len(dataloader)}, Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(dataloader)
    return avg_loss

  📂 Checkpoint geladen: Epoche 2, Loss 2.7889
✓ Loss: CrossEntropyLoss
✓ Optimizer: Adam (lr=0.0003)


In [18]:
NUM_EPOCHS = 10

for epoch in range(NUM_EPOCHS):
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    
    avg_loss = train_epoch(model, dataloader, criterion, optimizer, device)
    
    print(f"→ Average loss: {avg_loss:.4f}\n")

        
    if (epoch + 1) % 2 == 0:  # Every 2 epochs
        save_checkpoint(model, optimizer, epoch + 1, avg_loss, f'checkpoint_epoch_{epoch + 1}.pt')
    
    # Test generation after every epoch
    print("  Text Generation Test:")
    test_prompts = ["Die Geschichte", "Im Jahr", "Deutschland ist"]  # German prompts - the model only speaks German
    
    for start_text in test_prompts:
        start_tokens = encoding.encode(start_text)
        input_ids = torch.tensor([start_tokens], device=device)
        
        generated_ids = model.generate(input_ids, max_new_tokens=20, temperature=0.8)
        generated_text = encoding.decode(generated_ids[0].cpu().tolist())
        
        print(f"  '{generated_text}'")
    
    print(f"  '{generated_text}'\n")
    print("-" * 60)

Epoche 1/10
  Batch 10/8636, Loss: 2.6939
  Batch 20/8636, Loss: 2.8191
  Batch 30/8636, Loss: 2.9771
  Batch 40/8636, Loss: 2.7031
  Batch 50/8636, Loss: 2.8427
  Batch 60/8636, Loss: 2.7325
  Batch 70/8636, Loss: 2.7712
  Batch 80/8636, Loss: 3.1247
  Batch 90/8636, Loss: 2.9707
  Batch 100/8636, Loss: 2.8428
  Batch 110/8636, Loss: 2.8226
  Batch 120/8636, Loss: 2.9776
  Batch 130/8636, Loss: 2.9599
  Batch 140/8636, Loss: 3.0191
  Batch 150/8636, Loss: 3.0565
  Batch 160/8636, Loss: 3.0714
  Batch 170/8636, Loss: 3.3131
  Batch 180/8636, Loss: 2.8073
  Batch 190/8636, Loss: 3.0491
  Batch 200/8636, Loss: 3.1988
  Batch 210/8636, Loss: 2.9402
  Batch 220/8636, Loss: 3.0380
  Batch 230/8636, Loss: 3.0561
  Batch 240/8636, Loss: 3.3371
  Batch 250/8636, Loss: 3.1312
  Batch 260/8636, Loss: 3.2028
  Batch 270/8636, Loss: 2.9115
  Batch 280/8636, Loss: 3.0260
  Batch 290/8636, Loss: 2.7518
  Batch 300/8636, Loss: 2.9399
  Batch 310/8636, Loss: 3.1159
  Batch 320/8636, Loss: 2.9859
  Bat

KeyboardInterrupt: 

In [19]:
save_checkpoint(model, optimizer, epoch + 1, avg_loss, f'checkpoint_epoch_{epoch + 1}.pt')

  💾 Checkpoint gespeichert: checkpoint_epoch_3.pt


In [20]:
test_prompts = ["Die Geschichte", "Im Jahr", "Deutschland ist"]  # German prompts - the model only speaks German

for start_text in test_prompts:
    start_tokens = encoding.encode(start_text)
    input_ids = torch.tensor([start_tokens], device=device)
    
    generated_ids = model.generate(input_ids, max_new_tokens=50, temperature=0.8)
    generated_text = encoding.decode(generated_ids[0].cpu().tolist())
    
    print(f"  '{generated_text}'\n\n------------\n")

  'Die Geschichte der DDR vor. Sie wurde durch die Landesversicherung ersetzt, dass die Bundesrepublik Deutschland gewählt wurde. Nach einer Zeit der Bundesrep'

------------

  'Im Jahr 1362 entstanden. Ihr Somit wurde 1683 von Adolf von Geburt unter Hans Christian Wilhelm von Wietrich 1787 bis 1878 die letzte Klasse des Bistums Ritterfeldes mit'

------------

  'Deutschland ist der Deutsche Rechtsprechung; das Bundesamt für Wirtschaft und Verfassung ausführt das Ausführt, dass die Bundesrepublik Deutschland'

------------

